帮我写一个code block把所有created_at为2024年以后的name以" "为split，最后-1的项目变成seller_name。比如如下： name: 翻盖外缝线深蓝金羊 19开 闲鱼好划算验货宝 变成 name: 翻盖外缝线深蓝金羊 19开 seller_name: 闲鱼好划算验货宝 快速把mongodb中所有的做此修改

In [3]:
from datetime import datetime, timezone
from pymongo import MongoClient
import os

# ===== Mongo 连接 =====
MONGO_URI = os.getenv("MONGODB_URI", "mongodb://localhost:27017/")
DB_NAME = os.getenv("INV_DB", "inventory")   # ⚠️ 改成你实际 DB 名
COLL_NAME = "items"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
items = db[COLL_NAME]

# ===== 时间阈值：2024-01-01 =====
cutoff = datetime(2024, 1, 1, tzinfo=timezone.utc)

# ===== 查询条件：只处理 2024+ 且 seller_name 为空 =====
query = {
    "created_at": {"$gte": cutoff},
    "name": {"$regex": r"\s"},  # name 里至少有空格
    "$or": [
        {"seller_name": {"$exists": False}},
        {"seller_name": ""},
        {"seller_name": None},
    ]
}

cursor = items.find(query)
count = items.count_documents(query)
print(f"Found {count} items to process")

updated = 0

for it in cursor:
    _id = it.get("_id")
    name = (it.get("name") or "").strip()
    if not name:
        continue

    parts = name.split()
    if len(parts) < 2:
        continue

    new_seller = parts[-1].strip()
    new_name = " ".join(parts[:-1]).strip()

    if not new_seller or not new_name:
        continue

    print(f"[UPDATE] _id={_id} sku={it.get('sku','-')}")
    print(f"  OLD name: {name}")
    print(f"  NEW name: {new_name}")
    print(f"  seller_name: {new_seller}")

    items.update_one(
        {"_id": _id},
        {
            "$set": {
                "name": new_name,
                "seller_name": new_seller,
                "updated_at": datetime.now(timezone.utc),
            },
            "$push": {
                "audit": {
                    "at": datetime.now(timezone.utc),
                    "action": "SPLIT_NAME_TO_SELLER",
                    "detail": {
                        "from": name,
                        "to_name": new_name,
                        "seller_name": new_seller,
                    }
                }
            }
        }
    )

    updated += 1

print(f"Done. Updated {updated} items.")


Found 4008 items to process
[UPDATE] _id=693ef17b6c501f4dac249b2a sku=YEQNKBQ
  OLD name: 三格全黑牛 15bo0220链条 闲鱼梦缘验货宝
  NEW name: 三格全黑牛 15bo0220链条
  seller_name: 闲鱼梦缘验货宝
[UPDATE] _id=693ef17b6c501f4dac249b2b sku=ZBEKB5W
  OLD name: 三格全黑牛 18ma0241链条 闲鱼兔兔验货宝
  NEW name: 三格全黑牛 18ma0241链条
  seller_name: 闲鱼兔兔验货宝
[UPDATE] _id=693ef17b6c501f4dac249b2c sku=MG2CL9H
  OLD name: 五格深红银羊 96ma0141 闲鱼sami验货宝
  NEW name: 五格深红银羊 96ma0141
  seller_name: 闲鱼sami验货宝
[UPDATE] _id=693ef17a6c501f4dac249476 sku=C5AQBZL
  OLD name: maxi黑金荔枝 13开 珊瑚
  NEW name: maxi黑金荔枝 13开
  seller_name: 珊瑚
[UPDATE] _id=693ef17a6c501f4dac249477 sku=KJQE56L
  OLD name: 托特33黑金牛 18开有卡 香玉
  NEW name: 托特33黑金牛 18开有卡
  seller_name: 香玉
[UPDATE] _id=693ef17a6c501f4dac249478 sku=8K5FU8X
  OLD name: cf小号黑金荔枝v纹 30开有卡尘袋 香玉
  NEW name: cf小号黑金荔枝v纹 30开有卡尘袋
  seller_name: 香玉
[UPDATE] _id=693ef17a6c501f4dac249479 sku=QMB28HU
  OLD name: jumbo黑金荔枝单盖 13开有卡尘袋 芋头
  NEW name: jumbo黑金荔枝单盖 13开有卡尘袋
  seller_name: 芋头
[UPDATE] _id=693ef17a6c501f4dac24947a sku

ok，现在帮我写一个很小的code block，把目前数据库中所有的INBOUND状态的货物都换成REPARING

不需要更改我的codebase，请把代码直接以test发给我就行

In [4]:
from pymongo import MongoClient

# 按你的环境修改这两个
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "inventory"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
items = db["items"]

res = items.update_many(
    {"status": "INBOUND"},
    {"$set": {"status": "REPARING"}}
)

print(f"matched={res.matched_count}, modified={res.modified_count}")

matched=208, modified=208


我希望items里加入所有的sales的细节。并且希望完全删除sales这个collection

另外，我可能需要一个code block来帮我操作（返回string即可，我找地方跑），完成这个迁移。并且在这一系列操作之后，完全删除sales的collection可以对我的整个app不产生任何影响。 

In [5]:
from datetime import datetime, timezone
from bson import ObjectId
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"   # 改成你的
DB_NAME = "inventory"                     # 改成你的

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

items = db["items"]
sales = db["sales"]

def now_utc():
    return datetime.now(timezone.utc)

migrated = 0
skipped = 0

for s in sales.find({}):
    item_id = (s.get("item_id") or "").strip()
    if not item_id:
        skipped += 1
        continue

    try:
        oid = ObjectId(item_id)
    except Exception:
        skipped += 1
        continue

    it = items.find_one({"_id": oid})
    if not it:
        skipped += 1
        continue

    set_fields = {
        "buyer": s.get("buyer", "") or "",
        "channel": s.get("channel", "") or "",
        "sale_note": s.get("note", "") or "",
        "updated_at": it.get("updated_at") or now_utc(),
    }

    # sold_at：如果 items 没有，就用 sales 的
    if not it.get("sold_at") and s.get("sold_at"):
        set_fields["sold_at"] = s.get("sold_at")

    # 这些本来 items 已经有；如果缺失则补齐（可选）
    if it.get("sell_price", 0) in (None, 0) and s.get("sell_price"):
        set_fields["sell_price"] = int(s.get("sell_price") or 0)
    if not it.get("sell_currency") and s.get("sell_currency"):
        set_fields["sell_currency"] = s.get("sell_currency")
    if it.get("profit", 0) in (None, 0) and s.get("profit") is not None:
        set_fields["profit"] = int(s.get("profit") or 0)
    if not it.get("profit_currency") and s.get("profit_currency"):
        set_fields["profit_currency"] = s.get("profit_currency")

    audit_entry = {
        "at": now_utc(),
        "action": "MIGRATE_SALES_TO_ITEMS",
        "detail": {"sales_id": str(s.get("_id"))},
        "item_id": str(oid),
        "sku": it.get("sku"),
        "by": {"user_id": None, "username": "migration_script", "role": "Admin"},
        "req": {"method": "SCRIPT", "path": "migration", "ip": "", "ua": ""},
    }

    items.update_one(
        {"_id": oid},
        {"$set": set_fields, "$push": {"audit": audit_entry}}
    )
    migrated += 1

print("migrated:", migrated, "skipped:", skipped)

# 最后：彻底删除 sales collection
db.drop_collection("sales")
print("dropped collection: sales")

migrated: 4638 skipped: 0
dropped collection: sales



_id
693ef17a6c501f4dac2493fe
name
"cf小号蓝金羊 25开盒子"
brand
"Chanel"
currency
"RMB"
cost
18000
sell_price
5300
sell_currency
"SGD"
profit
10243
profit_currency
"RMB"
fee
0
status
"SOLD"
note
""
serial_code
"25104234"
code
""
accessories
""
seller_name
"佳佳"
seller_contact
""
source_type
"BUY_IN"
is_buy_in
true
is_consignment
false
created_at
2025-11-11T00:00:00.000+00:00
updated_at
2025-12-16T07:10:23.233+00:00
received_at
2025-11-11T00:00:00.000+00:00
sold_at
2025-12-16T07:10:10.000+00:00

audit
Array (4)

0
Object

1
Object
at
2025-12-14T17:49:23.163+00:00
action
"SPLIT_NAME_TO_SELLER"

detail
Object

2
Object
at
2025-12-16T07:10:23.233+00:00
action
"SOLD"

detail
Object
sell_price
5300
sell_currency
"SGD"
profit
10243
profit_currency
"RMB"
sold_at
2025-12-16T07:10:10.000+00:00

3
Object
at
2025-12-16T07:23:45.587+00:00
action
"MIGRATE_SALES_TO_ITEMS"

detail
Object
item_id
"693ef17a6c501f4dac2493fe"
sku
"KQZHDYJ"

by
Object

req
Object
sku
"KQZHDYJ"
buyer
"DSB USDT"
channel
"WA"
sale_note
""

你看下这是最新sold的一个item。其中有几个点我觉得可能会混淆。
1. 这个currency从目前的逻辑上应该是收货/寄卖的currency，更准确的命名应该是cost_currency。这个变量名需要改变。
2. 这里出现了两个sell price。外围的sell_price其实指的是商品的挂出价，也就是listing_price。还有相应的listing_currency。这两个变量名也需要改变。
3. 在item这个层级，应该加一个sold_price, sold_currency和。
4. 在audit这个层级，sell_price和sell_currency也应该更名为sold_price和sold_currency。

请帮我修改代码，确保所有的数据分析，charts，还有整个app的逻辑根据新的变量名来进行计算。

同时，请输出一个code block (string给我就行，我找地方跑），帮我把目前数据库中所有以上提到的变量名进行修改，以无缝对接更改后的代码。 

In [6]:
from datetime import datetime, timezone
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"   # 改成你的
DB_NAME = "inventory"                     # 改成你的

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

items = db["items"]
audit_logs = db.get_collection("audit_logs")

def _now():
    return datetime.now(timezone.utc)

def migrate_items():
    changed = 0
    for it in items.find({}, {"_id": 1, "status": 1, "currency": 1, "cost_currency": 1,
                              "sell_price": 1, "sell_currency": 1,
                              "listing_price": 1, "listing_currency": 1,
                              "sold_price": 1, "sold_currency": 1,
                              "audit": 1}):
        set_ops = {}
        unset_ops = {}

        # 1) currency -> cost_currency
        if "cost_currency" not in it and it.get("currency") is not None:
            set_ops["cost_currency"] = it.get("currency")
            unset_ops["currency"] = 1
        elif "currency" in it:
            # 已有 cost_currency 也把旧字段清掉，避免混淆
            unset_ops["currency"] = 1

        # 2) sell_* -> listing_*
        if "listing_price" not in it and it.get("sell_price") is not None:
            set_ops["listing_price"] = it.get("sell_price")
            unset_ops["sell_price"] = 1
        elif "sell_price" in it:
            unset_ops["sell_price"] = 1

        if "listing_currency" not in it and it.get("sell_currency") is not None:
            set_ops["listing_currency"] = it.get("sell_currency")
            unset_ops["sell_currency"] = 1
        elif "sell_currency" in it:
            unset_ops["sell_currency"] = 1

        # 3) SOLD items: fill sold_* if missing
        status = (it.get("status") or "").upper()
        if status == "SOLD":
            if it.get("sold_price") is None:
                lp = set_ops.get("listing_price", it.get("listing_price"))
                set_ops["sold_price"] = int(lp or 0)
            if it.get("sold_currency") is None:
                lc = set_ops.get("listing_currency", it.get("listing_currency"))
                cc = set_ops.get("cost_currency", it.get("cost_currency"))
                set_ops["sold_currency"] = (lc or cc or "SGD")

        # 4) rename audit.detail.sell_* -> audit.detail.sold_*
        aud = it.get("audit") or []
        aud_changed = False
        new_aud = []
        for a in aud:
            if not isinstance(a, dict):
                new_aud.append(a)
                continue
            d = a.get("detail")
            if isinstance(d, dict) and ("sell_price" in d or "sell_currency" in d):
                nd = dict(d)
                if "sell_price" in nd and "sold_price" not in nd:
                    nd["sold_price"] = nd.pop("sell_price")
                else:
                    nd.pop("sell_price", None)
                if "sell_currency" in nd and "sold_currency" not in nd:
                    nd["sold_currency"] = nd.pop("sell_currency")
                else:
                    nd.pop("sell_currency", None)
                a2 = dict(a)
                a2["detail"] = nd
                new_aud.append(a2)
                aud_changed = True
            else:
                new_aud.append(a)

        if aud_changed:
            set_ops["audit"] = new_aud

        if set_ops or unset_ops:
            update = {}
            if set_ops:
                update["$set"] = set_ops
            if unset_ops:
                update["$unset"] = unset_ops
            items.update_one({"_id": it["_id"]}, update)
            changed += 1

    print("items changed:", changed)

def migrate_audit_logs():
    if audit_logs is None:
        print("audit_logs: not found, skip")
        return
    changed = 0
    q = {"$or": [{"detail.sell_price": {"$exists": True}}, {"detail.sell_currency": {"$exists": True}}]}
    for r in audit_logs.find(q, {"_id": 1, "detail": 1}):
        d = r.get("detail") or {}
        if not isinstance(d, dict):
            continue
        set_ops = {}
        unset_ops = {}
        if "sell_price" in d:
            if "sold_price" not in d:
                set_ops["detail.sold_price"] = d.get("sell_price")
            unset_ops["detail.sell_price"] = 1
        if "sell_currency" in d:
            if "sold_currency" not in d:
                set_ops["detail.sold_currency"] = d.get("sell_currency")
            unset_ops["detail.sell_currency"] = 1

        if set_ops or unset_ops:
            update = {}
            if set_ops:
                update["$set"] = set_ops
            if unset_ops:
                update["$unset"] = unset_ops
            audit_logs.update_one({"_id": r["_id"]}, update)
            changed += 1

    print("audit_logs changed:", changed)

if __name__ == "__main__":
    migrate_items()
    migrate_audit_logs()
    print("done")

items changed: 5506
audit_logs changed: 0
done


ok。接下来，我希望修改整个audit的逻辑。首先，audit分两种，一种是Change_status，另一种是Change_details。

这两种操作都会作为array的一个item记录进入audit，但是数据结构稍有不同。

Change_Details指的是修改任何item信息的行为，包括编码，配件，邮寄单号，商品名称，品牌，成本，币种。。。等任何一切操作。
假如操作员在商品细节中同时更新了多项内容且只点了一次“保存”，这些内容也应该作为单独的entry保存在audit中。

对于Change_details操作，我理想的audit array中每个entry的结构如下：
at: xxx (时间）
action: Change_details
sku: xxx
target: xxx (被改的那个属性，比如e.g. accessories)
from: xxx (改之前的内容，如果空或者当前没有这个target就空着）
to: xxx （改之后的内容）
by: {user_id}

而Change_status指的是修改item状态的行为。这个我们记录的内容可以简单一些。
at: xxx (时间）
action: Change_status
sku: xxx
from: xxx
to: xxx
by: {user_id}

请记住，如果在同一次保存中修改了多项内容，则应该记录多项Change_details。另外，目前的menu中的 “操作历史” 页面也应该根据新的数据结构做修改。

最后，请帮我写一个code block（以string的形式返还给我，我自己找地方跑）来帮我清除目前所有的audit，以配合统一后续的audit数据格式。


In [7]:
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"   # 改成你的
DB_NAME = "inventory"                     # 改成你的

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

# 1) 清空 items.audit
res1 = db["items"].update_many({}, {"$unset": {"audit": 1}})
print("items.audit unset:", res1.modified_count)

# 2) 清空全局操作历史
db.drop_collection("audit_logs")
print("dropped collection: audit_logs")

items.audit unset: 5506
dropped collection: audit_logs


很好，我现在打算对开单的逻辑进行修改。

首先，售出开单界面（sale_new.html）和逻辑需要调整。删除“利润”和“利润币种”。添加“付款方式”（为下拉菜单，包含选项“Paynow”, "Credit Card", "Website payment", "Cash"，“Trade In”)。添加收据编号，自动填入{date}_{sku}（格式为2025-12-24_XG48LE这种)。这样既留给操作员更改的空间，也自动生成了receipt_no。另外，“备注”改成“销售单备注”，因为这里的variable name是sale_note，所以我们在文字上也要体现以做区分。

记住，所有在表格中不体现的选项一律不更新mongodb，所以利润和利润币种将不会更新。未来的操作逻辑就是开单的时候不更新利润，利润只有在后续老板统一在商品细节页面自己更新写入。

最终，所有开了销售单的items，在数据库中都应该会更新以下字段：sold_at, buyer, sale_note, sold_currency, sold_price, sale_channel（原名为channel）。这些字段应该被放入一个一级array字段中的一个array item。一级字段应该名为：sold_record。

请你再帮我写一个code block把目前所有existing销售数据变成这种新的数据格式。原本那些status为Sold的items中的这些销售相关的一级字段现在应该被放入sold_record中作为一个array item存在。请帮我写code block输出text即可，我自己找地方运行，

In [1]:
from datetime import timedelta, timezone, datetime
from pymongo import MongoClient

# 1) 修改为你的 MongoDB 连接（如需账号密码/云端地址就在这里改）
client = MongoClient("mongodb://localhost:27017")
db = client["inventory"]
items = db["items"]

tz8 = timezone(timedelta(hours=8))

def date_yyyy_mm_dd(dt):
    if not dt:
        return datetime.now(tz8).strftime("%Y-%m-%d")
    if isinstance(dt, str):
        # 如果你库里是 datetime 类型，这里不会走到；字符串就尽量直接截取
        return dt[:10]
    try:
        return dt.astimezone(tz8).strftime("%Y-%m-%d")
    except Exception:
        return datetime.now(tz8).strftime("%Y-%m-%d")

q = {
    "status": "SOLD",
    "$or": [
        {"sold_record": {"$exists": False}},
        {"sold_record": None},
        {"sold_record": []},
    ],
}

n = 0
for it in items.find(q, projection=None):
    sku = it.get("sku") or ""
    sold_at = it.get("sold_at")
    buyer = it.get("buyer") or ""
    sale_note = it.get("sale_note") or ""
    sold_currency = (it.get("sold_currency") or "").strip()
    sold_price = it.get("sold_price")
    sale_channel = it.get("sale_channel") or it.get("channel") or ""
    payment_method = it.get("payment_method") or ""
    receipt_no = it.get("receipt_no") or f"{date_yyyy_mm_dd(sold_at)}_{sku}" if sku else date_yyyy_mm_dd(sold_at)

    record = {
        "sold_at": sold_at,
        "buyer": buyer,
        "sale_note": sale_note,
        "sold_currency": sold_currency,
        "sold_price": sold_price,
        "sale_channel": sale_channel,
        "payment_method": payment_method,
        "receipt_no": receipt_no,
    }

    # 同步补齐一些新顶层字段（不删除旧 channel，保证兼容）
    set_fields = {}
    if it.get("sale_channel") is None and sale_channel:
        set_fields["sale_channel"] = sale_channel
    if it.get("receipt_no") is None and receipt_no:
        set_fields["receipt_no"] = receipt_no
    if it.get("payment_method") is None and payment_method:
        set_fields["payment_method"] = payment_method

    update_doc = {"$push": {"sold_record": record}}
    if set_fields:
        update_doc["$set"] = set_fields

    items.update_one({"_id": it["_id"]}, update_doc)
    n += 1

print("backfilled sold_record count:", n)

backfilled sold_record count: 4687


哦对了，既然更新了新的数据结构，那你帮我写个codeblock把之前数据结构留下的那些一级字段删除吧。这样只保留一个record。

另外，基于新的数据结构，请你帮我适配整个app的各处细节，包括数据分析部分的数据读取逻辑。要做到即使删除掉之前保留的一级字段，整个app仍然可以无缝运行。

In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["inventory"]

def cleanup(coll_name: str):
    coll = db[coll_name]
    res = coll.update_many(
        {},
        {"$unset": {
            # legacy / top-level sale fields to remove
            "sold_at": 1,
            "sold_price": 1,
            "sold_currency": 1,
            "buyer": 1,
            "channel": 1,         # old name
            "sale_channel": 1,    # new top-level name we no longer keep
            "sale_note": 1,
            "payment_method": 1,
            "receipt_no": 1,
        }}
    )
    print(coll_name, "matched:", res.matched_count, "modified:", res.modified_count)

cleanup("items")

# 如果你也希望 outsider 数据库同样只保留 sold_record（可选）
# cleanup("items_outsider")

items matched: 5563 modified: 5563
